# Chroma Model Walkthrough
George's vague attempt at understanding and explaining what's going on inside Chroma...

In [2]:
from chroma import Chroma, Protein, conditioners
from chroma.models import graph_classifier, procap
from chroma.utility.api import register_key
from chroma.utility.chroma import letter_to_point_cloud, plane_split_protein

import torch
import warnings

# When running this yourself, register your own API key (which you can get at https://chroma-weights.generatebiomedicines.com/):
#register_key("your_key_here")

# Initialise the model
chroma = Chroma()

# Make a protein
protein, trajectories = chroma.sample(full_output=True, chain_lengths=[50])

# Print the protein
print(protein)

# Print the output dictionary
print(trajectories)

Chroma(
  (backbone_network): GraphBackbone(
    (encoders): ModuleList(
      (0): BackboneEncoderGNN(
        (feature_graph): ProteinFeatureGraph(
          (graph_builder): ProteinGraph(
            (distances): Distances()
            (knn): kNN()
          )
          (node_layers): ModuleList(
            (0): NodeInternalCoords(
              (internal_coords): InternalCoords()
            )
          )
          (edge_layers): ModuleList(
            (0): EdgeDistance2mer(
              (layer_2mers): Edge2mers()
              (layer_distance): Distances()
              (rbf_function): RBFExpansion()
            )
            (1): EdgeOrientation2mer(
              (layer_2mers): Edge2mers()
            )
            (2): EdgeOrientationChain()
            (3): EdgeDistanceChain()
          )
          (node_linears): ModuleList(
            (0): Linear(in_features=20, out_features=512, bias=True)
          )
          (edge_linears): ModuleList(
            (0): Linear(in_fea

Now we have a protein.

In [33]:
output_filename = "protein.cif"
display(protein)
display(output_dict["trajectory"])
protein.to_CIF(output_filename)
trajectories["trajectory"].to_CIF(output_filename.replace(".cif", "_trajectory.cif"))



NGLWidget()

NGLWidget(max_frame=199)

NameError: name 'trajectories' is not defined

And a pretty video of it's generation trajectory.

Let's look at the model architecture.

In [30]:
# Print the model architecture
print(chroma)


Chroma(
  (backbone_network): GraphBackbone(
    (encoders): ModuleList(
      (0): BackboneEncoderGNN(
        (feature_graph): ProteinFeatureGraph(
          (graph_builder): ProteinGraph(
            (distances): Distances()
            (knn): kNN()
          )
          (node_layers): ModuleList(
            (0): NodeInternalCoords(
              (internal_coords): InternalCoords()
            )
          )
          (edge_layers): ModuleList(
            (0): EdgeDistance2mer(
              (layer_2mers): Edge2mers()
              (layer_distance): Distances()
              (rbf_function): RBFExpansion()
            )
            (1): EdgeOrientation2mer(
              (layer_2mers): Edge2mers()
            )
            (2): EdgeOrientationChain()
            (3): EdgeDistanceChain()
          )
          (node_linears): ModuleList(
            (0): Linear(in_features=20, out_features=512, bias=True)
          )
          (edge_linears): ModuleList(
            (0): Linear(in_fea

That's a lot of layers. (This output is also saved as chroma_model.txt)

Chroma builds a joint distribution of the sequence and and all-atom structure of protein complexes via the factorization:  
$$\log p(x,s,χ) = \log p(x) + \log p(s|x) + \log p(χ|x,s)  $$
= backbone likelihood + sequence likelihood given backbone + side-chain likelihood given backbone and sequence.  

We model these likelihoods with two networks:  
A backbone network trained as a diffusion model to model p(x)  
And a design network which models sequence and side chain chains conditioned on backbone structure.  

Both networks are based on a common graph neural network architecture, and we visualize the overall system in Supplementary Figure 7.  

![Chroma architecture](assets/sup_fig_7.png "Chroma architecture")  

We list important hyperparameters for the backbone network in Supplementary Table 2  

![Backbone network hyperparameters](assets/sup_tab_2.png "Backbone network hyperparameters")



In [31]:
# We can access bits of the model like this:
print(chroma.backbone_network.encoders[0].feature_graph.node_layers[0].internal_coords)

InternalCoords()


# Backbone Network

The backbone network starts with: encoder (BackboneEncoderGNN (FeatureGraph, GraphNN with 12 layers)).

AI: This takes the input protein structure and encodes it into a latent space.

Then it has: backbone_updates (GraphBackboneUpdate).

AI: This takes the output of the encoder and refines it.

Then it has:  time_features (FourierFeaturization)

Then: noise_perturb (DiffusionChainCov)

Then: loss_diffusion (ReconstructionLosses)

Then: mlp_W (MLP)

Let's have a deeper look at the BackboneEncoderGNN.

## BackboneEncoderGNN

### ProteinFeatureGraph

Graph_builder, Node_layers, Edge_layers, Node_linears, Edge_linears


### GraphNN (12 layers)

This is the update algorithm for each layer of the GNN:  

![GraphNN_Layer](assets/sup_alg_3.png "GNN Layer")



TypeError: 'Protein' object is not subscriptable